# Semantic Search with Transformers 🔎

In [1]:
# Run this cell beforehand so you do not see any warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os, subprocess

print("=== Environment ===")
print(f"LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', 'NOT SET')}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'NOT SET')}")
print(f"PATH: {os.environ.get('PATH', '')[:200]}")

print("\n=== nvidia-smi ===")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "FAILED")

print("\n=== libcuda.so ===")
print(subprocess.run(["find", "/usr/lib64", "/usr/lib", "/usr/local", "-name", "libcuda.so*", "-maxdepth", "3"], capture_output=True, text=True).stdout or "NOT FOUND")

print("\n=== PyTorch ===")
import torch
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.backends.cudnn.enabled: {torch.backends.cudnn.enabled}")


=== Environment ===
LD_LIBRARY_PATH: /soft/cuda-12.4/lib64
CUDA_VISIBLE_DEVICES: 0
PATH: /beegfs/general/oe24aam/semantic-search/venv/bin:/soft/cuda-12.4/bin:/usr/local/bin:/usr/local/sbin:/usr/local/bin:/usr/local/sbin:/beegfs/general/oe24aam/semantic-search/venv/bin:/home2/oe24aam/.mini

=== nvidia-smi ===
GPU 0: NVIDIA A100 80GB PCIe (UUID: GPU-44272749-ea27-3424-d91d-3d8c62c0be6f)


=== libcuda.so ===
/usr/lib64/libcuda.so
/usr/lib64/libcuda.so.1
/usr/lib64/libcuda.so.590.44.01


=== PyTorch ===
torch.version.cuda: 12.4
torch.cuda.is_available(): True
torch.backends.cudnn.enabled: True


## Import the Libraries

In [3]:
import pickle
import pandas as pd
import torch
import numpy as np
import faiss
import os

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn import preprocessing

## Load the Data

In [5]:
df = pd.read_json("data/research_papers.json")

In [6]:
df = df.drop(["author", "link", "tag"], axis=1)
df.head()

,day,id,month,summary,title,year
0,1,1802.00209v1,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018
1,12,1603.03827v1,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016
2,2,1606.00776v2,6,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016
3,23,1705.08142v2,5,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017
4,7,1709.02349v2,9,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017


In [7]:
print(f"Number of research papers: {len(df)}")
pd.set_option('display.max_colwidth', None)
df.head()

Number of research papers: 41000


,day,id,month,summary,title,year
0,1,1802.00209v1,2,"We propose an architecture for VQA which utilizes recurrent layers to\ngenerate visual and textual attention. The memory characteristic of the\nproposed recurrent attention units offers a rich joint embedding of visual and\ntextual features and enables the model to reason relations between several\nparts of the image and question. Our single model outperforms the first place\nwinner on the VQA 1.0 dataset, performs within margin to the current\nstate-of-the-art ensemble model. We also experiment with replacing attention\nmechanisms in other state-of-the-art models with our implementation and show\nincreased accuracy. In both cases, our recurrent attention mechanism improves\nperformance in tasks requiring sequential or relational reasoning on the VQA\ndataset.",Dual Recurrent Attention Units for Visual Question Answering,2018
1,12,1603.03827v1,3,"Recent approaches based on artificial neural networks (ANNs) have shown\npromising results for short-text classification. However, many short texts\noccur in sequences (e.g., sentences in a document or utterances in a dialog),\nand most existing ANN-based systems do not leverage the preceding short texts\nwhen classifying a subsequent one. In this work, we present a model based on\nrecurrent neural networks and convolutional neural networks that incorporates\nthe preceding short texts. Our model achieves state-of-the-art results on three\ndifferent datasets for dialog act prediction.",Sequential Short-Text Classification with Recurrent and Convolutional\n Neural Networks,2016
2,2,1606.00776v2,6,"We introduce the multiresolution recurrent neural network, which extends the\nsequence-to-sequence framework to model natural language generation as two\nparallel discrete stochastic processes: a sequence of high-level coarse tokens,\nand a sequence of natural language tokens. There are many ways to estimate or\nlearn the high-level coarse tokens, but we argue that a simple extraction\nprocedure is sufficient to capture a wealth of high-level discourse semantics.\nSuch procedure allows training the multiresolution recurrent neural network by\nmaximizing the exact joint log-likelihood over both sequences. In contrast to\nthe standard log- likelihood objective w.r.t. natural language tokens (word\nperplexity), optimizing the joint log-likelihood biases the model towards\nmodeling high-level abstractions. We apply the proposed model to the task of\ndialogue response generation in two challenging domains: the Ubuntu technical\nsupport domain, and Twitter conversations. On Ubuntu, the model outperforms\ncompeting approaches by a substantial margin, achieving state-of-the-art\nresults according to both automatic evaluation metrics and a human evaluation\nstudy. On Twitter, the model appears to generate more relevant and on-topic\nresponses according to automatic evaluation metrics. Finally, our experiments\ndemonstrate that the proposed model is more adept at overcoming the sparsity of\nnatural language and is better able to capture long-term structure.",Multiresolution Recurrent Neural Networks: An Application to Dialogue\n Response Generation,2016
3,23,1705.08142v2,5,"Multi-task learning is motivated by the observation that humans bring to bear\nwhat they know about related problems when solving new ones. Similarly, deep\nneural networks can profit from related tasks by sharing parameters with other\nnetworks. However, humans do not consciously decide to transfer knowledge\nbetween tasks. In Natural Language Processing (NLP), it is hard to predict if\nsharing will lead to improvements, particularly if tasks are only loosely\nrelated. To overcome this, we introduce Sluice Networks, a general framework\nfor multi-task learning where trainable parameters control the amount of\nsharing. Our framework generalizes previous proposals in enabling sharing of\nall combinations of subspaces, layers, and skip connections. We perform\nexperiments on three ta

## Retrieve the Model

In [8]:
# model source: @ https://huggingface.co/sentence-transformers/models
model = SentenceTransformer('all-mpnet-base-v2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"model: {model}\n device: {device}")

model: SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
 device: cuda


## Generate or Load the Embeddings

In [ ]:
# One-Off: Generate embeddings on a GPU-enabled environment preferably
embeddings = model.encode(df.summary.to_list(), show_progress_bar=True)
with open('data/new_embeddings.pickle', 'wb') as pkl:
    pickle.dump(embeddings, pkl)

In [9]:
# method to load embeddings
def load_embeddings(file_path:str, mode: str = 'rb'):
    with open(file_path, mode) as f:
        embeddings = pickle.load(f)
        return embeddings, len(embeddings), embeddings.shape

In [10]:
embeddings, length, shape = load_embeddings('data/new_embeddings.pickle')
print(f"embeddings: {embeddings[0]}\n length: {length}\n shape: {shape}")
print(f"Is instance of numpy arrays :{isinstance(embeddings, np.ndarray)}")

embeddings: [-1.09981090e-01  1.64143533e-01  6.77780509e-01  5.53457066e-02
 -5.35667650e-02  3.31018686e-01  3.55746090e-01 -4.42226738e-01
 -1.04353294e-01 -1.33925796e+00 -6.32903799e-02  9.93281364e-01
 -4.75986838e-01  2.11563110e-01  2.54530787e-01  2.63086498e-01
  1.14939690e+00 -1.39734179e-01 -1.43783987e-01  8.60710144e-02
  9.56531525e-01  1.09346494e-01 -1.67078674e-01  6.77421331e-01
 -6.76141977e-02 -4.19909060e-02  5.63696623e-01  9.84705448e-01
  4.20293331e-01 -1.94894403e-01  2.00043529e-01 -8.29738140e-01
 -2.93100595e-01 -1.27244681e-01  3.93143475e-01  8.00249875e-01
 -3.63039076e-01 -4.19261277e-01 -3.02978337e-01 -7.83954263e-01
  2.23565817e-01  3.50907624e-01 -2.59091035e-02  1.58726871e-01
 -6.72376871e-01  3.67020607e-01 -3.11773658e-01  7.09332347e-01
 -8.87612462e-01 -5.24798334e-01 -4.38391268e-01 -2.48129278e-01
 -2.39898264e-01 -3.71125698e-01  3.91665623e-02 -9.86732095e-02
 -3.75992149e-01 -1.50885075e-01  7.70348907e-01 -2.40745023e-02
 -6.37123585e

In [11]:
print(f"first paper: {embeddings[0].shape}")

first paper: (768,)


## Data Preparation and Helper Methods

In [12]:
label_encoder = preprocessing.LabelEncoder()
print(f"Data type before encoding: {df['id'].dtype}")
df.head()


Data type before encoding: object


,day,id,month,summary,title,year
0,1,1802.00209v1,2,"We propose an architecture for VQA which utilizes recurrent layers to\ngenerate visual and textual attention. The memory characteristic of the\nproposed recurrent attention units offers a rich joint embedding of visual and\ntextual features and enables the model to reason relations between several\nparts of the image and question. Our single model outperforms the first place\nwinner on the VQA 1.0 dataset, performs within margin to the current\nstate-of-the-art ensemble model. We also experiment with replacing attention\nmechanisms in other state-of-the-art models with our implementation and show\nincreased accuracy. In both cases, our recurrent attention mechanism improves\nperformance in tasks requiring sequential or relational reasoning on the VQA\ndataset.",Dual Recurrent Attention Units for Visual Question Answering,2018
1,12,1603.03827v1,3,"Recent approaches based on artificial neural networks (ANNs) have shown\npromising results for short-text classification. However, many short texts\noccur in sequences (e.g., sentences in a document or utterances in a dialog),\nand most existing ANN-based systems do not leverage the preceding short texts\nwhen classifying a subsequent one. In this work, we present a model based on\nrecurrent neural networks and convolutional neural networks that incorporates\nthe preceding short texts. Our model achieves state-of-the-art results on three\ndifferent datasets for dialog act prediction.",Sequential Short-Text Classification with Recurrent and Convolutional\n Neural Networks,2016
2,2,1606.00776v2,6,"We introduce the multiresolution recurrent neural network, which extends the\nsequence-to-sequence framework to model natural language generation as two\nparallel discrete stochastic processes: a sequence of high-level coarse tokens,\nand a sequence of natural language tokens. There are many ways to estimate or\nlearn the high-level coarse tokens, but we argue that a simple extraction\nprocedure is sufficient to capture a wealth of high-level discourse semantics.\nSuch procedure allows training the multiresolution recurrent neural network by\nmaximizing the exact joint log-likelihood over both sequences. In contrast to\nthe standard log- likelihood objective w.r.t. natural language tokens (word\nperplexity), optimizing the joint log-likelihood biases the model towards\nmodeling high-level abstractions. We apply the proposed model to the task of\ndialogue response generation in two challenging domains: the Ubuntu technical\nsupport domain, and Twitter conversations. On Ubuntu, the model outperforms\ncompeting approaches by a substantial margin, achieving state-of-the-art\nresults according to both automatic evaluation metrics and a human evaluation\nstudy. On Twitter, the model appears to generate more relevant and on-topic\nresponses according to automatic evaluation metrics. Finally, our experiments\ndemonstrate that the proposed model is more adept at overcoming the sparsity of\nnatural language and is better able to capture long-term structure.",Multiresolution Recurrent Neural Networks: An Application to Dialogue\n Response Generation,2016
3,23,1705.08142v2,5,"Multi-task learning is motivated by the observation that humans bring to bear\nwhat they know about related problems when solving new ones. Similarly, deep\nneural networks can profit from related tasks by sharing parameters with other\nnetworks. However, humans do not consciously decide to transfer knowledge\nbetween tasks. In Natural Language Processing (NLP), it is hard to predict if\nsharing will lead to improvements, particularly if tasks are only loosely\nrelated. To overcome this, we introduce Sluice Networks, a general framework\nfor multi-task learning where trainable parameters control the amount of\nsharing. Our framework generalizes previous proposals in enabling sharing of\nall combinations of subspaces, layers, and skip connections. We perform\nexperiments on three ta

In [13]:
df['encoded_id'] = label_encoder.fit_transform(df['id'])
print(f"Data type after encoding: {df['encoded_id'].dtype}")
df.head()

Data type after encoding: int64


,day,id,month,summary,title,year,encoded_id
0,1,1802.00209v1,2,"We propose an architecture for VQA which utilizes recurrent layers to\ngenerate visual and textual attention. The memory characteristic of the\nproposed recurrent attention units offers a rich joint embedding of visual and\ntextual features and enables the model to reason relations between several\nparts of the image and question. Our single model outperforms the first place\nwinner on the VQA 1.0 dataset, performs within margin to the current\nstate-of-the-art ensemble model. We also experiment with replacing attention\nmechanisms in other state-of-the-art models with our implementation and show\nincreased accuracy. In both cases, our recurrent attention mechanism improves\nperformance in tasks requiring sequential or relational reasoning on the VQA\ndataset.",Dual Recurrent Attention Units for Visual Question Answering,2018,36693
1,12,1603.03827v1,3,"Recent approaches based on artificial neural networks (ANNs) have shown\npromising results for short-text classification. However, many short texts\noccur in sequences (e.g., sentences in a document or utterances in a dialog),\nand most existing ANN-based systems do not leverage the preceding short texts\nwhen classifying a subsequent one. In this work, we present a model based on\nrecurrent neural networks and convolutional neural networks that incorporates\nthe preceding short texts. Our model achieves state-of-the-art results on three\ndifferent datasets for dialog act prediction.",Sequential Short-Text Classification with Recurrent and Convolutional\n Neural Networks,2016,18198
2,2,1606.00776v2,6,"We introduce the multiresolution recurrent neural network, which extends the\nsequence-to-sequence framework to model natural language generation as two\nparallel discrete stochastic processes: a sequence of high-level coarse tokens,\nand a sequence of natural language tokens. There are many ways to estimate or\nlearn the high-level coarse tokens, but we argue that a simple extraction\nprocedure is sufficient to capture a wealth of high-level discourse semantics.\nSuch procedure allows training the multiresolution recurrent neural network by\nmaximizing the exact joint log-likelihood over both sequences. In contrast to\nthe standard log- likelihood objective w.r.t. natural language tokens (word\nperplexity), optimizing the joint log-likelihood biases the model towards\nmodeling high-level abstractions. We apply the proposed model to the task of\ndialogue response generation in two challenging domains: the Ubuntu technical\nsupport domain, and Twitter conversations. On Ubuntu, the model outperforms\ncompeting approaches by a substantial margin, achieving state-of-the-art\nresults according to both automatic evaluation metrics and a human evaluation\nstudy. On Twitter, the model appears to generate more relevant and on-topic\nresponses according to automatic evaluation metrics. Finally, our experiments\ndemonstrate that the proposed model is more adept at overcoming the sparsity of\nnatural language and is better able to capture long-term structure.",Multiresolution Recurrent Neural Networks: An Application to Dialogue\n Response Generation,2016,19318
3,23,1705.08142v2,5,"Multi-task learning is motivated by the observation that humans bring to bear\nwhat they know about related problems when solving new ones. Similarly, deep\nneural networks can profit from related tasks by sharing parameters with other\nnetworks. However, humans do not consciously decide to transfer knowledge\nbetween tasks. In Natural Language Processing (NLP), it is hard to predict if\nsharing will lead to improvements, particularly if tasks are only loosely\nrelated. To overcome this, we introduce Sluice Networks, a general framework\nfor multi-task learning where trainable parameters control the amount of\nsharing. Our framework generalizes previous proposals in enabling sharing of\nall combinations of subspaces, layers, and skip connections. We per

In [14]:
"""
Return a list of column values for papers specified by their IDs
parameters:
  df: The DataFrame in which the data is contained
  I: List of IDs of the papers for which the information is required
  column: Column of the DataFrame where the required information is stored
"""
def id_to_info(df, I, column):
    print(f"df: {df}\n I: {I}\n column: {column}")
    return [list(df[df['encoded_id'] == idx][column]) for idx in I]

## Set up the Index

In [15]:
embeddings_np = np.array(embeddings, dtype=np.float32)
print(f"Numpy embeddings: {embeddings_np}\n data type: {embeddings.dtype}")

Numpy embeddings: [[-0.10998109  0.16414353  0.6777805  ... -0.01285609  0.01391327
  -0.59528315]
 [-0.18171197 -0.0667079   0.28978541 ... -0.15399022 -0.47588152
  -0.03911386]
 [-0.20065093 -0.21362196  0.5258912  ...  0.03071197 -0.7731148
   0.49049348]
 ...
 [-0.60289073 -0.00454931  0.84874785 ...  0.01303321 -0.4628397
  -0.33787972]
 [-1.2161096   0.07581758  0.3547052  ...  0.07892998 -0.3391658
   1.00243   ]
 [-0.30350107  0.08865086  0.5019363  ...  0.17140785 -0.5170257
   0.31726295]]
 data type: float32


In [16]:
def create_gpu_index(embedding_dim):
    num_gpus = faiss.get_num_gpus()
    print(f"Faiss detected {num_gpus} GPU(s)")

    if num_gpus == 0:
        print("No GPU available for Faiss, falling back to CPU index")
        index = faiss.IndexFlatL2(embedding_dim)
        return faiss.IndexIDMap(index)

    # Always use device 0 — SLURM's CUDA_VISIBLE_DEVICES already remaps
    # physical GPUs so the app always sees device 0
    res = faiss.StandardGpuResources()
    config = faiss.GpuIndexFlatConfig()
    config.device = 0

    gpu_index = faiss.GpuIndexFlatL2(res, embedding_dim, config)
    return faiss.IndexIDMap(gpu_index)

In [17]:
gpu_index_map = create_gpu_index(embeddings_np.shape[1])

Faiss detected 1 GPU(s)


In [18]:
gpu_index_map.add_with_ids(
    embeddings_np, df["encoded_id"][:length].values.astype("int64")
)

print(f"Number of embeddings in the Faiss index: {gpu_index_map.ntotal}")

Number of embeddings in the Faiss index: 41000


In [19]:
## Search with a Summary
df.iloc[1337, [3, 1]]

summary    In this paper we study the application of convolutional neural networks for\njointly detecting objects depicted in still images and estimating their 3D\npose. We identify different feature representations of oriented objects, and\nenergies that lead a network to learn this representations. The choice of the\nrepresentation is crucial since the pose of an object has a natural, continuous\nstructure while its category is a discrete variable. We evaluate the different\napproaches on the joint object detection and pose estimation task of the\nPascal3D+ benchmark using Average Viewpoint Precision. We show that a\nclassification approach on discretized viewpoints achieves state-of-the-art\nperformance for joint object detection and pose estimation, and significantly\noutperforms existing baselines on this benchmark.
id                                                                                                                                                                     

In [20]:
# Search by existing paper summary
D, I = gpu_index_map.search(np.array([embeddings[1337]]), k=10)
pd.DataFrame({'L2 distance': D.flatten().tolist(), 'ML paper IDs': I.flatten().tolist(), 'ML paper titles': id_to_info(df, I.flatten(), 'title'), 'Summaries': id_to_info(df, I.flatten(), 'summary')}).head(10)

df:        day            id  month  \
0        1  1802.00209v1      2   
1       12  1603.03827v1      3   
2        2  1606.00776v2      6   
3       23  1705.08142v2      5   
4        7  1709.02349v2      9   
...    ...           ...    ...   
40995   18   1404.4702v2      4   
40996   22   1404.5421v1      4   
40997   22   1404.5899v1      4   
40998   25   1404.6369v1      4   
40999   27   1407.0380v1      6   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

,L2 distance,ML paper IDs,ML paper titles,Summaries
0,0.000031,12964,[Convolutional Neural Networks for joint object detection and pose\n estimation: A comparative study],"[In this paper we study the application of convolutional neural networks for\njointly detecting objects depicted in still images and estimating their 3D\npose. We identify different feature representations of oriented objects, and\nenergies that lead a network to learn this representations. The choice of the\nrepresentation is crucial since the pose of an object has a natural, continuous\nstructure while its category is a discrete variable. We evaluate the different\napproaches on the joint object detection and pose estimation task of the\nPascal3D+ benchmark using Average Viewpoint Precision. We show that a\nclassification approach on discretized viewpoints achieves state-of-the-art\nperformance for joint object detection and pose estimation, and significantly\noutperforms existing baselines on this benchmark.]"
1,46.787537,23635,[Beyond Holistic Object Recognition: Enriching Image Understanding with\n Part States],"[Important high-level vision tasks such as human-object interaction, image\ncaptioning and robotic manipulation require rich semantic descriptions of\nobjects at part level. Based upon previous work on part localization, in this\npaper, we address the problem of inferring rich semantics imparted by an object\npart in still images. We propose to tokenize the semantic space as a discrete\nset of part states. Our modeling of part state is spatially localized,\ntherefore, we formulate the part state inference problem as a pixel-wise\nannotation problem. An iterative part-state inference neural network is\nspecifically designed for this task, which is efficient in time and accurate in\nperformance. Extensive experiments demonstrate that the proposed method can\neffectively predict the semantic states of parts and simultaneously correct\nlocalization errors, thus benefiting a few visual understanding applications.\nThe other contribution of this paper is our part state dataset which contains\nrich part-level semantic annotations.]"
2,55.466721,15599,[Maximum-Margin Structured Learning with Deep Networks for 3D Human Pose\n Estimation],"[This paper focuses on structured-output learning using deep neural networks\nfor 3D human pose estimation from monocular images. Our network takes an image\nand 3D pose as inputs and outputs a score value, which is high when the\nimage-pose pair matches and low otherwise. The network structure consists of a\nconvolutional neural network for image feature extraction, followed by two\nsub-networks for transforming the image features and pose into a joint\nembedding. The score function is then the dot-product between the image and\npose embeddings. The image-pose embedding and score function are jointly\ntrained using a maximum-margin cost function. Our proposed framework can be\ninterpreted as a special form of structured support vector machines where the\njoint feature space is discriminatively learned using deep neural networks. We\ntest our framework on the Human3.6m dataset and obtain state-of-the-art results\ncompared to other recent methods. Finally, we present visualizations of the\nimage-pose embedding space, demonstrating the network has learned a high-level\nembedding of body-orientation and pose-configuration.]"
3,56.263046,11454,[Articulated Pose Estimation by a Graphical Model with Image Dependent\n Pairwise Relations],"[We present a method for estimating articulated human pose from a single\nstatic image based on a graphical model with novel pairwise relations that make\nadaptive use of local image measurements. More precisely, we specify a\ngraphical model for human pose which exploits the fact the local image\nmeasurements can be used both to detect parts (or joints) and also to predict\nthe spatial relationships between them (Image Dependent Pairwise Relations).\nThese spatial relationships are represent

## Prompt Search


In [21]:
query = "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data"

In [22]:
embed = model.encode([query])

print(f"embed shape: {embed.shape}")
print(f"embed type: {type(embed)}")

embed shape: (1, 768)
embed type: <class 'numpy.ndarray'>


In [23]:

D, I = gpu_index_map.search(embed.astype("float32"), k=10)

results = {'L2 distances':D.flatten().tolist(), 'ML paper IDs':I.flatten().tolist(), "Titles": id_to_info(df, I.flatten(), 'title'), "Summaries": id_to_info(df, I.flatten(), 'summary')}

pd.DataFrame(results).head(10)

df:        day            id  month  \
0        1  1802.00209v1      2   
1       12  1603.03827v1      3   
2        2  1606.00776v2      6   
3       23  1705.08142v2      5   
4        7  1709.02349v2      9   
...    ...           ...    ...   
40995   18   1404.4702v2      4   
40996   22   1404.5421v1      4   
40997   22   1404.5899v1      4   
40998   25   1404.6369v1      4   
40999   27   1407.0380v1      6   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

,L2 distances,ML paper IDs,Titles,Summaries
0,121.829422,17329,[Feature Representation for ICU Mortality],"[Good predictors of ICU Mortality have the potential to identify high-risk\npatients earlier, improve ICU resource allocation, or create more accurate\npopulation-level risk models. Machine learning practitioners typically make\nchoices about how to represent features in a particular model, but these\nchoices are seldom evaluated quantitatively. This study compares the\nperformance of different representations of clinical event data from MIMIC II\nin a logistic regression model to predict 36-hour ICU mortality. The most\ncommon representations are linear (normalized counts) and binary (yes/no).\nThese, along with a new representation termed ""hill"", are compared using both\nL1 and L2 regularization. Results indicate that the introduced ""hill""\nrepresentation outperforms both the binary and linear representations, the hill\nrepresentation thus has the potential to improve existing models of ICU\nmortality.]"
1,128.814255,9072,[A Model of the Mechanisms Underlying Exploratory Behaviour],"[A model of the mechanisms underlying exploratory behaviour, based on\nempirical research and refined using a computer simulation, is presented. The\nbehaviour of killifish from two lakes, one with killifish predators and one\nwithout, was compared in the laboratory. Plotting average activity in a novel\nenvironment versus time resulted in an inverted-U-shaped curve for both groups;\nhowever, the curve for killifish from the lake without predators was (1)\nsteeper, (2) reached a peak value earlier, (S) reached a higher peak value, and\n(4) subsumed less area than the curve for killifish from the lake with\npredators. We hypothesize that the shape of the exploration curve reflects a\ncompetition between motivational subsystems that excite and inhibit exploratory\nbehaviour in a way that is tuned to match the affordance probabilities of the\nanimal's environment. A computer implementation of this model produced curves\nwhich differed along the same four dimensions as differentiate the two\nkillifish curves. All four differences were reproduced in the model by tuning a\nsingle parameter: the time-dependent component of the decay-rate of the\nexploration-inhibiting subsystem.]"
2,129.733719,4285,[Gibbs Sampling in Open-Universe Stochastic Languages],"[Languages for open-universe probabilistic models (OUPMs) can represent\nsituations with an unknown number of objects and iden- tity uncertainty. While\nsuch cases arise in a wide range of important real-world appli- cations,\nexisting general purpose inference methods for OUPMs are far less efficient\nthan those available for more restricted lan- guages and model classes. This\npaper goes some way to remedying this deficit by in- troducing, and proving\ncorrect, a generaliza- tion of Gibbs sampling to partial worlds with possibly\nvarying model structure. Our ap- proach draws on and extends previous generic\nOUPM inference methods, as well as aux- iliary variable samplers for\nnonparametric mixture models. It has been implemented for BLOG, a well-known\nOUPM language. Combined with compile-time optimizations, the resulting\nalgorithm yields very substan- tial speedups over existing methods on sev- eral\ntest cases, and substantially improves the practicality of OUPM languages\ngenerally.]"
3,130.121780,31382,[The Voynich Manuscript is Written in Natural Language: The Pahlavi\n Hypothesis],"[The late medieval Voynich Manuscript (VM) has resisted decryption and was\nconsidered a meaningless hoax or an unsolvable cipher. Here, we provide\nevidence that the VM is written in natural language by establishing a relation\nof the Voynich alphabet and the Iranian Pahlavi script. Many of the Voynich\ncharacters are upside-down versions of their Pahlavi counterparts, which may be\nan effect of different writing directions. Other Voynich letters can be\nexplained as ligatures or departures from Pahlavi with